# Hotel Booking Cancellation Prediction

### Project Summary

Hotels frequently face revenue uncertainty due to reservation cancellations. Accurately predicting which reservations are likely to cancel can help hotels improve revenue forecasting, optimize overbooking strategies, and better allocate resources.

The goal of this project is to develop a predictive model that identifies reservations likely to cancel using historical hotel booking data. This analysis follows the CRISP-DM framework, including data exploration, data cleaning, modeling, and evaluation. The resulting insights may help hotel managers make more informed decisions regarding booking policies and customer management.

In [1]:
import pandas as pd

# load dataset
df = pd.read_csv("../data/hotel_bookings.csv")

# preview the data
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## Data & Data Dictionary

This dataset contains hotel booking records and includes reservation details, customer information, and whether the booking was canceled. The following cells preview the data structure and summarize the features included in the dataset.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

### Data Dictionary

| Feature | Data Type | Description |
|---|---|---|
| hotel | object | Type of hotel (Resort Hotel or City Hotel) |
| is_canceled | int64 | Whether the booking was canceled (1 = yes, 0 = no) |
| lead_time | int64 | Number of days between booking date and arrival date |
| arrival_date_year | int64 | Year of arrival |
| arrival_date_month | object | Month of arrival |
| arrival_date_week_number | int64 | Week number of arrival date |
| arrival_date_day_of_month | int64 | Day of the month of arrival |
| stays_in_weekend_nights | int64 | Number of weekend nights booked |
| stays_in_week_nights | int64 | Number of weekday nights booked |
| adults | int64 | Number of adults included in the booking |
| children | float64 | Number of children included in the booking |
| babies | int64 | Number of babies included in the booking |
| meal | object | Type of meal booked |
| country | object | Country of origin of the guest |
| market_segment | object | Market segment designation |
| distribution_channel | object | Booking distribution channel |
| is_repeated_guest | int64 | Whether the guest has booked before (1 = yes, 0 = no) |
| previous_cancellations | int64 | Number of previous bookings canceled by the customer |
| previous_bookings_not_canceled | int64 | Number of previous bookings not canceled by the customer |
| reserved_room_type | object | Code for room type originally reserved |
| assigned_room_type | object | Code for room type assigned at check-in |
| booking_changes | int64 | Number of changes made to the booking |
| deposit_type | object | Type of deposit made for the booking |
| agent | float64 | ID of the travel agent that made the booking |
| company | float64 | ID of the company/entity that made the booking |
| days_in_waiting_list | int64 | Number of days the booking was on the waiting list |
| customer_type | object | Type of customer |
| adr | float64 | Average daily rate |
| required_car_parking_spaces | int64 | Number of parking spaces required |
| total_of_special_requests | int64 | Number of special requests made by the customer |
| reservation_status | object | Reservation status at the end of the process |
| reservation_status_date | object | Date of the final reservation status |

## Selected Features for Exploratory Analysis

To limit the scope of exploratory analysis, four features were selected that are expected to have a strong relationship with reservation cancellations.

| Feature | Reason for Selection |
|-------|-------|
| lead_time | Longer booking lead times may increase the likelihood of cancellation |
| deposit_type | Deposit policies may influence customer commitment to reservations |
| previous_cancellations | Past customer behavior may indicate future cancellation risk |
| total_of_special_requests | Guests with more requests may be more committed to the reservation |

In [3]:
selected_features = [
    "lead_time",
    "deposit_type",
    "previous_cancellations",
    "total_of_special_requests",
    "is_canceled"
]

In [4]:
import pandas as pd

stats_rows = []

for col in selected_features:
    s = df[col]
    
    row = {
        "feature": col,
        "count_values": s.count(),
        "count_missing": s.isna().sum(),
        "unique_values": s.nunique(),
        "mode": s.mode().iloc[0] if not s.mode().empty else None
    }
    
    if pd.api.types.is_numeric_dtype(s):
        row.update({
            "min": s.min(),
            "Q1": s.quantile(0.25),
            "median": s.median(),
            "Q3": s.quantile(0.75),
            "max": s.max(),
            "mean": s.mean(),
            "std": s.std(),
            "skewness": s.skew(),
            "kurtosis": s.kurt()
        })
    else:
        row.update({
            "min": None,
            "Q1": None,
            "median": None,
            "Q3": None,
            "max": None,
            "mean": None,
            "std": None,
            "skewness": None,
            "kurtosis": None
        })
    
    stats_rows.append(row)

univariate_stats = pd.DataFrame(stats_rows)
univariate_stats

,feature,count_values,count_missing,unique_values,mode,min,Q1,median,Q3,max,mean,std,skewness,kurtosis
0,lead_time,119390,0,479,0,0.0,18.0,69.0,160.0,737.0,104.011416,106.863097,1.346550,1.696449
1,deposit_type,119390,0,3,No Deposit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,previous_cancellations,119390,0,15,0,0.0,0.0,0.0,0.0,26.0,0.087118,0.844336,24.458049,674.073693
3,total_of_special_requests,119390,0,6,0,0.0,0.0,0.0,1.0,5.0,0.571363,0.792798,1.349189,1.492565
4,is_canceled,119390,0,2,0,0.0,0.0,0.0,1.0,1.0,0.370416,0.482918,0.536678,-1.712005


In [5]:
univariate_stats.to_excel("../excel/hotel_univariate_statistics.xlsx", index=False)

## Numeric Relationships (Correlation)

In [6]:
import pandas as pd

numeric_features = [
    "lead_time",
    "previous_cancellations",
    "total_of_special_requests"
]

correlation_results = []

for col in numeric_features:
    corr = df[col].corr(df["is_canceled"])
    
    correlation_results.append({
        "feature": col,
        "pearson_correlation_with_is_canceled": corr
    })

correlation_df = pd.DataFrame(correlation_results)
correlation_df

,feature,pearson_correlation_with_is_canceled
0,lead_time,0.293123
1,previous_cancellations,0.110133
2,total_of_special_requests,-0.234658


## Categorical relationship (Chi-square)

In [7]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df["deposit_type"], df["is_canceled"])

chi2, p, dof, expected = chi2_contingency(contingency_table)

chi_square_result = pd.DataFrame({
    "feature": ["deposit_type"],
    "chi_square_statistic": [chi2],
    "p_value": [p]
})

chi_square_result

,feature,chi_square_statistic,p_value
0,deposit_type,27677.329241,0.0


In [8]:
bivariate_results = pd.concat([correlation_df, chi_square_result], ignore_index=True)
bivariate_results

,feature,pearson_correlation_with_is_canceled,chi_square_statistic,p_value
0,lead_time,0.293123,NaN,NaN
1,previous_cancellations,0.110133,NaN,NaN
2,total_of_special_requests,-0.234658,NaN,NaN
3,deposit_type,NaN,27677.329241,0.0


In [9]:
bivariate_results.to_excel("../excel/hotel_bivariate_statistics.xlsx", index=False)